# CMIP6 Winds

runs the reusable wind-extreme analysis code for historical vs future scenarios. Outputs include spatial change heat maps, historical/future histograms, scenario trend scatter plots, and summary significance tables.

Edit `wind_extremes.config.json` first. Replace any remaining `PLACEHOLDER_*` values with the real NetCDF variable, coordinate, and historical-file names.

Overall workflow is:   
find project folder -> add src to Python imports -> load config -> validate config -> run full workflow -> display summary table ->   
display generated figures -> optionally rerun with grid-cell significance -> optionally build wind speed from U/V files

In [ ]:
from pathlib import Path
import sys
#get the current working directory
cwd = Path.cwd().resolve()
#find where the tool folder might be
candidates = [cwd, cwd / "Wind_Extreme_Analysis_Tool"]
#also check parent folders
for parent in cwd.parents:
    candidates.extend([parent, parent / "Wind_Extreme_Analysis_Tool"])
#find the first folder that contains source package and config file
TOOL_ROOT = next(
    candidate for candidate in candidates
    if (candidate / "src" / "wind_extreme_analysis").exists()
    and (candidate / "wind_extremes.config.json").exists()
)
#make the project root the folder above the tool folder
PROJECT_ROOT = TOOL_ROOT.parent
#define where to pull source code from
SRC_DIR = TOOL_ROOT / "src"
#add src to import path so it imports without installing
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
#define config file this notebook will use
CONFIG_PATH = TOOL_ROOT / "wind_extremes.config.json"
#choose which dataset inside config.json to run
DATASET_NAME = "WSPD10"
#display paths so you can confirm the notebook found the right files
TOOL_ROOT, PROJECT_ROOT, CONFIG_PATH

## Check the Config

Run this before the full analysis. If any placeholders are still present, the next cell will list them.

In [ ]:
#import config.json helpers
from wind_extreme_analysis.config import get_dataset_config, load_config, validate_dataset_config
#load config into python dictionary
config = load_config(CONFIG_PATH)
#pull out the seleceted dataset config
dataset_name, dataset = get_dataset_config(config, DATASET_NAME)
#check for missing values or placeholders
issues = validate_dataset_config(dataset)

if issues:
    print("Config values to fix:")
    for issue in issues:
        print(f"- {issue}")
else:
    print("Config looks ready.")

## What the Significance Tests Mean

For each future scenario period, the workflow compares the 98th and 99.9th percentile against the 1995-2014 historical baseline

The bootstrap confidence interval is the main test. If the interval for `future percentile - historical percentile` does not include zero, the percentile change is flagged as significant

The nearby timesteps for 3_hourly data are not truly independent. The config uses block bootstrap resampling by default: `block_bootstrap_timesteps = 56`, which is 7 days of 3-hourly data

The Mann-Whitney and Kolmogorov-Smirnov p-values are additional regional distribution checks. Check README for further descriptions on those. They test whether the overall distribution changed, not specifically whether the 98th or 99.9th percentile changed

## Run the Analysis

This writes PNG maps/plots and `summary_statistics.csv` to the output folder listed in the config

In [ ]:
from wind_extreme_analysis.workflow import analyze_dataset

if issues:
    raise ValueError("Fix config placeholders before running the analysis.")

summary = analyze_dataset(config, dataset_name=dataset_name, base_dir=PROJECT_ROOT)
summary

## Review the Generated Figures

In [ ]:

from IPython.display import Image, display
#builds the ouput folder path for this dataset
output_dir = PROJECT_ROOT / config["output_dir"] / dataset_name
for figure_path in sorted(output_dir.glob("*.png")):
    print(figure_path.name)
    display(Image(filename=str(figure_path)))

## Optional: Grid-Cell Significance Maps

Grid-cell bootstrap maps are useful, but they can be slow on 5 km data. Turn them on only after the basic workflow is working. The output NetCDF files contain absolute change, percent change, confidence interval bounds, p-value, and a significant/not-significant mask for each grid cell.

In [ ]:
# Uncomment to run grid-cell significance maps.
# config["make_grid_significance_maps"] = True
# config["bootstrap_iterations"] = 500
# summary_with_grid_tests = analyze_dataset(config, dataset_name=dataset_name, base_dir=PROJECT_ROOT)
# summary_with_grid_tests

## Optional: Build Wind Speed From U/V Components

Use this when future files contain component winds instead of speed magnitude. Replace the variable placeholders before running.

In [ ]:
# from wind_extreme_analysis.uv_to_speed import combine_uv_netcdf
#
# combine_uv_netcdf(
#     u_file=PROJECT_ROOT / "Analysis" / "U10_BCC-CSM2-MR_ssp585_2040-2059_3hourly.nc",
#     v_file=PROJECT_ROOT / "Analysis" / "V10_BCC-CSM2-MR_ssp585_2040-2059_3hourly.nc",
#     u_var="PLACEHOLDER_U_VAR",
#     v_var="PLACEHOLDER_V_VAR",
#     output_file=PROJECT_ROOT / "Analysis" / "WSPD10_BCC-CSM2-MR_ssp585_2040-2059_3hourly.nc",
#     speed_var="PLACEHOLDER_WIND_SPEED_VAR",
# )